Project Structure
- Setup Chunking and processing steps for inputs (PDFs, docx, txt)
- Setup PGVector Store (VS) + postgresql => Done
- Using LLMs (OpenAI, Gemini,...) to query vectors from VS

# Create PostgreDB 

In [3]:
import sys
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

True

In [4]:
import psycopg2

db_name = os.environ['POSTGRES_DB']
host = "localhost"
password = os.environ['POSTGRES_PASSWORD']
port = "5432"
user = os.environ['POSTGRES_USER']
# conn = psycopg2.connect(connection_string)
conn = psycopg2.connect(
    dbname=db_name,
    host=host,
    password=password,
    port=port,
    user=user,
)
conn.autocommit = True

# with conn.cursor() as c:
#     c.execute(f"DROP DATABASE IF EXISTS {db_name}")
#     c.execute(f"CREATE DATABASE {db_name}")

ModuleNotFoundError: No module named 'psycopg2'

In [5]:
print(conn.status)
print(conn.get_dsn_parameters())
with conn.cursor() as cursor:
    cursor.execute("SELECT 1")
    print("Connection is active")

1
{'user': 'admin', 'channel_binding': 'prefer', 'dbname': 'rag_db', 'host': 'localhost', 'port': '5432', 'options': '', 'sslmode': 'prefer', 'sslcompression': '0', 'sslcertmode': 'allow', 'sslsni': '1', 'ssl_min_protocol_version': 'TLSv1.2', 'gssencmode': 'prefer', 'krbsrvname': 'postgres', 'gssdelegation': '0', 'target_session_attrs': 'any', 'load_balance_hosts': 'disable'}
Connection is active


# Checking file input

In [1]:
from pydantic import BaseModel, Field, field_validator
from pydantic_ai import Agent, RunContext
from uuid import UUID
from chonkie import TokenChunker, SemanticChunker

In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from pathlib import Path
import PyPDF2
from chonkie import SemanticChunker
from sentence_transformers import SentenceTransformer


def extract_text_from_pdf(pdf_path):
    """
    Extract text from PDF file using PyPDF2.

    Args:
        pdf_path (str): Path to the PDF file

    Returns:
        str: Extracted text from all pages
    """
    text = ""
    with open(pdf_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)

        for page_num in range(len(pdf_reader.pages)):
            page = pdf_reader.pages[page_num]
            text += page.extract_text() + "\n"

    return text


def chunk_with_semantic_chunker(text, chunk_size=512, similarity_threshold=0.7, embedding_model=None):
    """
    Chunk text using Chonkie's SemanticChunker with custom embedding model.

    Args:
        text (str): Text to chunk
        chunk_size (int): Maximum tokens per chunk
        similarity_threshold (float): Similarity threshold for semantic chunking
        embedding_model: Custom embedding model (SentenceTransformer or similar)

    Returns:
        list: List of chunks
    """
    if embedding_model:
        chunker = SemanticChunker(
            chunk_size=chunk_size,
            similarity_threshold=similarity_threshold,
            embedding_model=embedding_model
        )
    else:
        # Use default embedding model
        chunker = SemanticChunker(
            chunk_size=chunk_size,
            similarity_threshold=similarity_threshold
        )

    chunks = chunker.chunk(text)
    return chunks


def save_chunks_to_file(chunks, output_path, chunker_type):
    """
    Save chunks to a text file.

    Args:
        chunks (list): List of chunks
        output_path (str): Output file path
        chunker_type (str): Type of chunker used
    """
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(f"Chunks created using {chunker_type}\n")
        f.write("=" * 50 + "\n\n")

        for i, chunk in enumerate(chunks, 1):
            f.write(f"Chunk {i}:\n")
            f.write("-" * 20 + "\n")
            f.write(f"{chunk.text}\n\n")
            f.write(f"Tokens: {chunk.token_count}\n")
            if hasattr(chunk, 'start_index'):
                f.write(f"Start Index: {chunk.start_index}\n")
            if hasattr(chunk, 'end_index'):
                f.write(f"End Index: {chunk.end_index}\n")
            f.write("\n" + "="*50 + "\n\n")

/usr/local/Caskroom/miniconda/base/envs/project/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
print("\nStep 1: Loading documents")
# texts = extract_text_from_pdf(r'D:\Project\rag_with_llama\docs\llama2.pdf')
texts = extract_text_from_pdf(r'../docs/llama2.pdf')

# Create output directory
output_dir = "chunked_output"
os.makedirs(output_dir, exist_ok=True)

print("\nStep 2: Chunking with SemanticChunker")
semantic_chunks = chunk_with_semantic_chunker(
    texts,
    chunk_size=512,
    similarity_threshold=0.7
    )


Step 1: Loading documents

Step 2: Chunking with SemanticChunker


/usr/local/Caskroom/miniconda/base/envs/project/lib/python3.12/site-packages/chonkie/embeddings/auto.py:87: UserWarning: Failed to load minishlab/potion-base-32M with Model2VecEmbeddings: model2vec is not available. Please install it via `pip install chonkie[model2vec]`
Falling back to loading default provider model.
  warnings.warn(
/usr/local/Caskroom/miniconda/base/envs/project/lib/python3.12/site-packages/chonkie/embeddings/auto.py:95: UserWarning: Failed to load the default model for Model2VecEmbeddings: model2vec is not available. Please install it via `pip install chonkie[model2vec]`
Falling back to SentenceTransformerEmbeddings.
  warnings.warn(


In [14]:
semantic_chunks[0]

Chunk(text='Llama 2 : Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗Louis Martin†Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
', token_count=43, start_index=0, end_index=167)

In [ ]:
from typing import List, Dict, Any, Optional

class EmbeddingGenerator:
    """Generate embeddings using SentenceTransformers."""

    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        """
        Initialize embedding generator.

        Args:
            model_name: SentenceTransformer model name
        """
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)
        self.embedding_dim = self.model.get_sentence_embedding_dimension()

    def embed_text(self, text: str) -> List[float]:
        """
        Generate embedding for a single text.
        Args:
            text: Input text
        Returns:
            List of embedding values
        """
        embedding = self.model.encode(text)
        return embedding.tolist()

    def embed_batch(self, texts: List[str]) -> List[List[float]]:
        """
        Generate embeddings for multiple texts.
        Args:
            texts: List of input texts
        Returns:
            List of embedding lists
        """
        embeddings = self.model.encode(texts)
        return [emb.tolist() for emb in embeddings]

embedding_model = 'all-MiniLM-L6-v2'
embedding_generator = EmbeddingGenerator(embedding_model)

In [16]:
embedded_txt = embedding_generator.embed_batch(semantic_chunks)

In [17]:
len(embedded_txt)

1332

In [1]:
import psycopg2
from pgvector.psycopg2 import register_vector  # Missing import

In [ ]:
# -- First, turn off the pager to avoid the 'more' error
# \pset pager off

# -- Now run your queries (notice the semicolon at the end)
# SELECT version();

# SELECT 2+2 AS result;

# SELECT NOW();

# SELECT 'Hello PostgreSQL!' AS message;

In [ ]:
# import sentence_transformers
# model = sentence_transformers.SentenceTransformer('all-MiniLM-L6-v2')
# model.save('./embedded_model/all-MiniLM-L6-v2')
# model_test = sentence_transformers.SentenceTransformer('./embedded_model/all-MiniLM-L6-v2')

# Test fulll flow

In [14]:
%reload_ext autoreload
%autoreload 2

In [1]:
import os
import uuid
from pathlib import Path
from typing import List, Dict, Any

from fastapi import FastAPI, File, UploadFile, HTTPException, Form
from fastapi.responses import HTMLResponse
from pydantic import BaseModel, Field

import sys
sys.path.append('../document_processing')  # Go up one level, then into folder

import sys
sys.path.append('../docs')  # Go up one level, then into folder

# Your existing components - unchanged
from embed_chunks_to_db import ChunkEmbeddingPipeline, EmbeddingGenerator, VectorStore
import google.generativeai as genai

/opt/miniconda3/envs/longnv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
db_params = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'port': os.getenv('DB_PORT', '5432'),
    'dbname': os.getenv('DB_NAME', 'rag_db'),
    'user': os.getenv('DB_USER', 'admin'),
    'password': os.getenv('DB_PASSWORD', 'admin')
}

table_name = 'document_chunks'

# Don't initialize models at startup
pipeline = None
def get_pipeline():
    global pipeline
    if pipeline is None:
        pipeline = ChunkEmbeddingPipeline(
                    db_params=db_params,
                    embedding_model='all-MiniLM-L6-v2',
                    table_name=table_name
                )
    return pipeline

pipeline = get_pipeline()
file = pipeline.extract_text_from_pdf(r'../docs/llama2.pdf')
chunks = pipeline.chunk_text(file)

In [14]:
def test_pgvector_connection():
    """Validate pgvector integration at the connection level"""
    vector_store = VectorStore(db_params, 'document_chunks')
    
    try:
        conn = vector_store._get_connection()
        with conn.cursor() as cur:
            # Test 1: Confirm pgvector types are registered
            cur.execute("SELECT '[1,2,3]'::vector;")
            result = cur.fetchone()[0]
            print(f"✓ Vector type working: {result}")
            
            # Test 2: Verify cosine distance operator
            cur.execute("SELECT '[1,0,0]'::vector <=> '[0,1,0]'::vector;")
            distance = cur.fetchone()[0]
            print(f"✓ Cosine distance: {distance:.3f}")
            
            # Test 3: Table schema validation
            cur.execute(f"""
                SELECT column_name, data_type 
                FROM information_schema.columns 
                WHERE table_name = '{vector_store.table_name}' 
                AND column_name = 'embedding'
            """)
            schema = cur.fetchone()
            print(f"✓ Embedding column: {schema}")
            
        conn.close()
        return True
        
    except Exception as e:
        print(f"✗ pgvector connection failed: {e}")
        return False
    
test_pgvector_connection()

✓ Vector type working: [1. 2. 3.]
✓ Cosine distance: 1.000
✓ Embedding column: ('embedding', 'USER-DEFINED')


True

In [15]:
from dataclasses import dataclass
from typing import List, Dict, Any, Optional


@dataclass
class Chunk:
    """Chunk data structure to match your existing interface."""
    id: str
    document_id: str
    text: str
    embedding: List[float]
    metadata: Optional[Dict] = None

In [16]:
# Generate embeddings in batch
print("Generating embeddings...")
embeddings = pipeline.embedding_generator.embed_batch([chunk.text for chunk in chunks])

Generating embeddings...


In [17]:
chunk_size = 512
similarity_threshold = 0.7
filename = 'llama2.pdf'
file_type = 'pdf'
file_size = len(file)
document_id = str(uuid.uuid4())
metadata = {'source': 'llama2.pdf'}

# Create Chunk objects using your interface
chunk_objects = []
for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
    chunk_metadata = {
        'chunk_index': i,
        'token_count': chunk.token_count,
        'start_index': getattr(chunk, 'start_index', None),
        'end_index': getattr(chunk, 'end_index', None),
        'chunk_size': chunk_size,
        'similarity_threshold': similarity_threshold,
        'embedding_model': pipeline.embedding_generator.model_name,
        'embedding_dimension': len(embedding),
        'filename': filename,
        'file_type': file_type,
        'file_size': file_size
    }

    # Add any additional metadata
    if metadata:
        chunk_metadata.update(metadata)

    chunk_obj = Chunk(
        id=str(uuid.uuid4()),
        document_id=document_id,
        text=chunk.text,
        embedding=embedding,
        metadata=chunk_metadata
    )
    chunk_objects.append(chunk_obj)

In [18]:
# Use your add_chunks method with pgvector
print("Inserting chunks into database using pgvector...")
pipeline.vector_store.add_chunks(chunk_objects)

Inserting chunks into database using pgvector...


In [19]:
test_results = pipeline.search_documents('text', limit=1, threshold=0.1)
assert len(test_results) > 0, "Insert succeeded but search failed"
print(f"✓ Search operational: {len(test_results)} results")

✓ Search operational: 1 results


In [ ]:
def fast_reset_vector_store(vector_store):
    """Two-step reset: TRUNCATE then DROP for optimal performance"""
    conn = vector_store._get_connection()
    
    try:
        with conn.cursor() as cur:
            # Step 1: Instant data removal (no WAL overhead)
            cur.execute(f"TRUNCATE TABLE {vector_store.table_name} CASCADE;")
            print("✓ Data truncated instantly")
            
            # Step 2: Clean schema removal
            cur.execute(f"DROP TABLE {vector_store.table_name} CASCADE;")
            print("✓ Table schema dropped")
            
        conn.commit()
        
    except Exception as e:
        conn.rollback()
        raise e
    finally:
        conn.close()

# # Usage
# fast_reset_vector_store(pipeline.vector_store)

✓ Data truncated instantly
✓ Table schema dropped


# Check inserting 

In [44]:
# Verify success
conn = pipeline.vector_store._get_connection()
with conn.cursor() as cur:
    cur.execute(f"SELECT COUNT(*) FROM {pipeline.vector_store.table_name}")
    count_after = cur.fetchone()[0]
    
    # Validate embedding integrity
    cur.execute(f"""
        SELECT count(*)
        FROM {pipeline.vector_store.table_name} 
    """)
    
    results = cur.fetchall()

UndefinedTable: relation "document_chunks" does not exist
LINE 1: SELECT COUNT(*) FROM document_chunks
                             ^


In [43]:
results[0][0]

2740

In [21]:
results[0]

('0cef73c0-a5ea-4b45-93f7-152110c773a0',
 '62fea6ed-85a6-464f-b0fd-5ec2402d0dd8',
 'Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen\n',
 array([ 7.12140650e-03,  1.58904716e-01, -4.11183424e-02, -1.60896499e-02,
        -1.19244307e-01,  1.66793726e-02,  7.66922235e-02,  2.09896639e-03,
        -7.99249765e-03, -3.11004985e-02,  6.69134455e-03, -1.19025603e-01,
         9.55456793e-02, -4.00688797e-02,  2.43884884e-03,  2.39558239e-02,
        -2.64738058e-03,  2.02801265e-02,  7.83016905e-03, -7.15690851e-02,
        -2.38060020e-02, -1.06638998e-01, -9.48443916e-03,  2.73222420e-02,
        -6.30895793e-02, -4.59778449e-03,  1.97286680e-02,  1.22988073e-03,
         2.02850271e-02, -3.98318060e-02, -3.29108462e-02,  1.30037352e-01,
         7.59432688e-02, -2.78772581e-02, -1.24753239e-02,  8.16547722e-02,
        -5.94117120e-02,  4.51860111e-03,  1.69892292e-02, -4.73478902e-03,
        -2.29553226e-02,  7.49923289e-03, -2.45867502e-02, -6.

In [46]:
import sys
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv('./deployment/.env')

False

In [34]:
# embedding_model = './embedded_model/all-MiniLM-L6-v2'
# embedding_generator = EmbeddingGenerator(embedding_model)
query = "What is Llama 2?"
results = pipeline.search_documents(
    query=query,
    limit=100,
    threshold=0.5
)

# Step 2: Build context for LLM
context = "\n\n".join([f"[Context {i+1}]: {r['text']}" for i, r in enumerate(results)])

# Step 3: Generate response with Gemini (if available)
gemini_key = os.getenv('GOOGLE_API_KEY')
genai.configure(api_key=gemini_key)
if gemini_key:
    model = genai.GenerativeModel('gemini-2.5-flash')
    prompt = f"""Answer based on this context:
            {context}
            Question: {query}
            Answer:"""
    
    response = model.generate_content(prompt)
    answer = response.text

E0000 00:00:1758732769.428342 1368407 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


In [35]:
answer

'Llama 2 is an updated version of Llama 1, trained on a new mix of publicly available data. It is a new technology and a Large Language Model (LLM) that was pretrained on 2 trillion tokens of data from publicly available sources. Llama 2 carries potential risks with its use.'

# Test with Pydantic AI

In [6]:
import sys
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv('../deployment/.env')

True

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.google import GoogleModel
from pydantic_ai.providers.google import GoogleProvider

from pydantic import BaseModel, Field
from typing import Dict, Any, List, Optional

class ResponseWithMetadata(BaseModel):
    content: str = Field(description="Main response")
    confidence: Optional[float] = Field(None, ge=0, le=1)
    word_count: Optional[int] = None
    complexity_level: Optional[str] = None
    custom_metadata: Dict[str, Any] = Field(default_factory=dict)

# Create agent with structured response
agent = Agent(model)

provider = GoogleProvider(api_key=os.getenv('GOOGLE_API_KEY'))
model = GoogleModel('gemini-2.5-flash', provider=provider)
agent = Agent(model, result_type=ResponseWithMetadata)

In [18]:
answer

AgentRunResult(output='**NLTK (Natural Language Toolkit)** is a powerful, open-source library written in **Python** that is widely used for working with human language data. In the context of text processing, it serves as a comprehensive platform for building Python programs to analyze, process, and understand natural language.\n\nThink of it as a Swiss Army knife for Natural Language Processing (NLP) tasks, especially for educational purposes, research, and rapid prototyping.\n\nHere\'s a breakdown of what NLTK is and its role in text processing:\n\n### What is NLTK?\n\n*   **Python Library:** It\'s a collection of modules and functions for Python.\n*   **NLP Focus:** Specifically designed for tasks related to Natural Language Processing, which is a subfield of artificial intelligence that deals with the interaction between computers and human (natural) languages.\n*   **Comprehensive:** It provides a vast array of algorithms and data sets (corpora) that support almost all facets of N

In [10]:
question = 'What is NLTK in context of text processing?'
answer = await agent.run(question)

# Logfire test

In [1]:
import sys
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv('../deployment/.env')

True

In [ ]:
import logfire

logfire_key = os.getenv('LOGFIRE_WRITE_TOKEN')
logfire.configure(token=logfire_key)

# Different log levels
logfire.trace("Trace level message")
logfire.debug("Debug message with {var}", var="value")
logfire.info("Hello, {name}!", name="world")
logfire.notice("Notice level")
logfire.warn("Warning message")
logfire.error("Error occurred")
logfire.fatal("Fatal error")

11:17:30.226 Hello, world!
11:17:30.226 Notice level
11:17:30.227 Warning message
11:17:30.227 Error occurred
11:17:30.227 Fatal error


Logfire project URL: ]8;id=223067;https://logfire-us.pydantic.dev/adrien-2025/rag-demo\https://logfire-us.pydantic.dev/adrien-2025/rag-demo]8;;\
Currently retrying 1 failed export(s)


In [5]:
import logfire
import time

logfire_key = os.getenv('LOGFIRE_WRITE_TOKEN')
logfire.configure(token=logfire_key)

# Top-level span
with logfire.span("Process Order", order_id="ORD-12345"):
    logfire.info("Starting order processing")
    
    # First child span
    with logfire.span("Validate Order"):
        logfire.debug("Checking order items")
        time.sleep(0.1)
        logfire.info("Order validation: {status}", status="passed")
    
    # Second child span
    with logfire.span("Calculate Total"):
        logfire.debug("Item count: {count}", count=3)
        time.sleep(0.05)
        total = 99.99
        logfire.info("Total calculated: ${amount}", amount=total)
    
    # Third child span
    with logfire.span("Process Payment"):
        logfire.info("Charging card ending in {last4}", last4="4242")
        time.sleep(0.2)
        logfire.info("Payment successful")
    
    logfire.info("Order completed successfully")

13:46:29.744 Process Order
13:46:29.744   Starting order processing
13:46:29.744   Validate Order
13:46:29.850     Order validation: passed
13:46:29.850   Calculate Total
13:46:29.906     Total calculated: $99.99
13:46:29.906   Process Payment
13:46:29.907     Charging card ending in 4242
13:46:30.112     Payment successful
13:46:30.113   Order completed successfully


Logfire project URL: ]8;id=463429;https://logfire-us.pydantic.dev/adrien-2025/rag-demo\https://logfire-us.pydantic.dev/adrien-2025/rag-demo]8;;\


# Test with ReRank

In [ ]:
import sys
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv('../deployment/.env')

import json
import uuid
import asyncpg
import numpy as np
import PyPDF2
from pathlib import Path
import asyncio

# Disable tokenizers parallelism warning
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Import your existing chunking functions
sys.path.append('/Users/longnv/Coding/rag_llama_index/document_processing')

from sentence_transformers import SentenceTransformer

from typing import List, Dict, Any, Optional
from dataclasses import dataclass
from chunk_pdf_with_chonkie import process_document


@dataclass
class Chunk:
    """Chunk data structure to match your existing interface."""
    id: str
    document_id: str
    text: str
    embedding: List[float]
    metadata: Optional[Dict] = None


class EmbeddingGenerator:
    """Generate embeddings using SentenceTransformers."""

    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        """
        Initialize embedding generator.

        Args:
            model_name: SentenceTransformer model name
        """
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)
        self.embedding_dim = self.model.get_sentence_embedding_dimension()

    def embed_text(self, text: str) -> List[float]:
        """
        Generate embedding for a single text.
        Args:
            text: Input text
        Returns:
            List of embedding values
        """
        embedding = self.model.encode(text)
        return embedding.tolist()

    def embed_batch(self, texts: List[str]) -> List[List[float]]:
        """
        Generate embeddings for multiple texts.
        Args:
            texts: List of input texts
        Returns:
            List of embedding lists
        """
        embeddings = self.model.encode(texts)
        return [emb.tolist() for emb in embeddings]


class VectorStore:
    """Vector store using pgvector for efficient similarity search."""

    def __init__(self, connection_params: Dict[str, str], table_name: str = "chunks"):
        """
        Initialize vector store with pgvector support.
        Args:
            connection_params: Database connection parameters
            table_name: Name of the chunks table
        """
        self.connection_params = connection_params
        self.table_name = table_name
        self.connection_string = self._build_connection_string()
        self._initialized = False

    def _build_connection_string(self) -> str:
        """Build asyncpg connection string from parameters."""
        return f"postgresql://{self.connection_params['user']}:{self.connection_params['password']}@{self.connection_params['host']}:{self.connection_params['port']}/{self.connection_params['dbname']}"

    async def _get_connection(self):
        """Get database connection with pgvector support."""
        conn = await asyncpg.connect(self.connection_string)
        # Enable pgvector extension
        await conn.execute("CREATE EXTENSION IF NOT EXISTS vector;")
        return conn

    async def _initialize_database(self):
        """Initialize database with pgvector extension and table."""
        try:
            if self._initialized:
                return

            conn = await self._get_connection()

            # Create table with proper vector column
            # Assuming 384-dimensional embeddings for all-MiniLM-L6-v2
            # Adjust dimension based on your model
            await conn.execute(f"""
            CREATE TABLE IF NOT EXISTS {self.table_name} (
                id TEXT PRIMARY KEY,
                document_id TEXT NOT NULL,
                text TEXT NOT NULL,
                embedding vector(384),  -- Adjust dimension as needed
                metadata JSONB,
                created_at TIMESTAMP WITH TIME ZONE DEFAULT CURRENT_TIMESTAMP
            );
            """)

            # Create index for similarity search
            await conn.execute(f"""
            CREATE INDEX IF NOT EXISTS {self.table_name}_embedding_idx
            ON {self.table_name} USING ivfflat (embedding vector_cosine_ops)
            WITH (lists = 100);
            """)

            # Create index on document_id for filtering
            await conn.execute(f"""
            CREATE INDEX IF NOT EXISTS {self.table_name}_document_id_idx
            ON {self.table_name} (document_id);
            """)

            await conn.close()
            self._initialized = True
            print(f"Database initialized with table: {self.table_name}")

        except Exception as e:
            print(f"Error initializing database: {e}")
            raise

    async def add_chunks(self, chunks: List[Chunk], batch_size: int = 100):
        """Add chunks to vector store using batch insert for efficiency."""
        try:
            if not self._initialized:
                await self._initialize_database()

            conn = await self._get_connection()

            # Prepare data for batch insert
            chunk_data = []
            for chunk in chunks:
                # Convert embedding list to proper pgvector format
                embedding_str = '[' + ','.join(map(str, chunk.embedding)) + ']'

                chunk_data.append((
                    chunk.id,
                    chunk.document_id,
                    chunk.text,
                    embedding_str,
                    json.dumps(
                        chunk.metadata) if chunk.metadata else json.dumps({})
                ))

            # Use asyncpg's executemany for efficient batch insert
            insert_sql = f"""
            INSERT INTO {self.table_name} (id, document_id, text, embedding, metadata)
            VALUES ($1, $2, $3, $4, $5)
            ON CONFLICT (id) DO UPDATE SET
                document_id = EXCLUDED.document_id,
                text = EXCLUDED.text,
                embedding = EXCLUDED.embedding,
                metadata = EXCLUDED.metadata;
            """

            # Process in batches
            for i in range(0, len(chunk_data), batch_size):
                batch = chunk_data[i:i + batch_size]
                await conn.executemany(insert_sql, batch)

            await conn.close()
            print(f"Added {len(chunks)} chunks to vector store")

        except Exception as e:
            print(f"Error adding chunks: {e}")
            raise

    async def search_similar_chunks(self, query_embedding: List[float],
                                    limit: int = 5, threshold: float = 0.7,
                                    document_ids: Optional[List[str]] = None) -> List[Dict]:
        """
        Search for similar chunks using pgvector cosine similarity.

        Args:
            query_embedding: Query embedding vector
            limit: Maximum number of results
            threshold: Similarity threshold (0-1, higher = more similar)
            document_ids: Optional list of document IDs to filter by

        Returns:
            List of similar chunks with metadata
        """
        try:
            if not self._initialized:
                await self._initialize_database()

            conn = await self._get_connection()

            # Build query with optional document filtering
            base_query = f"""
                SELECT
                    id,
                    text,
                    metadata,
                    document_id,
                    (1 - (embedding <=> $1)) as similarity
                FROM {self.table_name}
                WHERE (1 - (embedding <=> $1)) >= $2
            """

            # Convert query embedding to proper pgvector format
            query_embedding_str = '[' + ','.join(map(str, query_embedding)) + ']'
            params = [query_embedding_str, threshold]

            if document_ids:
                base_query += " AND document_id = ANY($3)"
                params.append(document_ids)
                base_query += """
                    ORDER BY embedding <=> $1
                    LIMIT $4
                """
                params.append(limit)
            else:
                base_query += """
                    ORDER BY embedding <=> $1
                    LIMIT $3
                """
                params.append(limit)

            rows = await conn.fetch(base_query, *params)

            results = []
            for row in rows:
                results.append({
                    'chunk_id': row['id'],
                    'text': row['text'],
                    'metadata': row['metadata'] if isinstance(row['metadata'], (dict, type(None))) else json.loads(row['metadata']),
                    'document_id': row['document_id'],
                    'similarity': float(row['similarity'])
                })

            await conn.close()
            return results

        except Exception as e:
            print(f"Error searching chunks: {e}")
            raise

    async def delete_document_chunks(self, document_id: str) -> int:
        """Delete all chunks for a specific document."""
        try:
            if not self._initialized:
                await self._initialize_database()

            conn = await self._get_connection()
            result = await conn.execute(
                f"DELETE FROM {self.table_name} WHERE document_id = $1", document_id)
            deleted_count = int(result.split()[-1]) if result else 0
            await conn.close()
            print(
                f"Deleted {deleted_count} chunks for document: {document_id}")
            return deleted_count
        except Exception as e:
            print(f"Error deleting document chunks: {e}")
            raise

    async def get_collection_stats(self) -> Dict[str, Any]:
        """Get statistics about the vector store."""
        try:
            if not self._initialized:
                await self._initialize_database()

            conn = await self._get_connection()
            row = await conn.fetchrow(f"""
            SELECT
                COUNT(*) as total_chunks,
                COUNT(DISTINCT document_id) as total_documents,
                AVG(LENGTH(text)) as avg_text_length,
                MIN(created_at) as earliest_chunk,
                MAX(created_at) as latest_chunk
            FROM {self.table_name}
            """)

            stats = {
                'total_chunks': row['total_chunks'],
                'total_documents': row['total_documents'],
                'avg_text_length': float(row['avg_text_length']) if row['avg_text_length'] else 0,
                'earliest_chunk': row['earliest_chunk'].isoformat() if row['earliest_chunk'] else None,
                'latest_chunk': row['latest_chunk'].isoformat() if row['latest_chunk'] else None
            }

            await conn.close()
            return stats
        except Exception as e:
            print(f"Error getting stats: {e}")
            raise


class ChunkEmbeddingPipeline:
    """Complete pipeline for chunking documents and storing embeddings."""

    def __init__(self, db_params: Dict[str, str], embedding_model: str,
                 table_name: str):
        """
        Initialize the pipeline.
        Args:
            db_params: Database connection parameters
            embedding_model: SentenceTransformer model name
            table_name: Name of the chunks table
        """
        self.embedding_generator = EmbeddingGenerator(embedding_model)
        self.vector_store = VectorStore(db_params, table_name)

    async def process_document(self, file_path: str, chunk_size: int = 512,
                               similarity_threshold: float = 0.5,
                               document_id: str = None, metadata: Dict = None) -> str:
        """
        Process a document: chunk using imported function, then embed and store.

        Args:
            file_path: Path to the document file
            chunk_size: Maximum tokens per chunk
            similarity_threshold: Similarity threshold for chunking
            document_id: Optional document ID (if None, will generate one)
            metadata: Additional metadata for the document

        Returns:
            Document ID
        """
        file_path = Path(file_path)
        filename = file_path.name
        file_type = file_path.suffix.lower().replace('.', '')
        file_size = file_path.stat().st_size

        # Generate document ID if not provided
        if document_id is None:
            document_id = str(uuid.uuid4())

        print(f"Processing document: {filename} (ID: {document_id})")

        # Use the imported process_document function for chunking with page numbers
        chunks = process_document(
            file_path=str(file_path),
            chunk_size=chunk_size,
            similarity_threshold=similarity_threshold,
            embedding_model=self.embedding_generator.model_name
        )

        print(f"Created {len(chunks)} chunks using imported chunking function")

        # Prepare chunks for embedding
        chunk_texts = [chunk.text for chunk in chunks]

        # Generate embeddings in batch
        print("Generating embeddings...")
        embeddings = self.embedding_generator.embed_batch(chunk_texts)

        # Create Chunk objects for database storage
        chunk_objects = []
        for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
            # Extract page number from chunk (set by imported process_document function)
            page_number = getattr(chunk, 'page_number', 1)

            chunk_metadata = {
                'chunk_index': i,
                'token_count': chunk.token_count,
                'start_index': getattr(chunk, 'start_index', None),
                'end_index': getattr(chunk, 'end_index', None),
                'page_number': page_number,
                'chunk_size': chunk_size,
                'similarity_threshold': similarity_threshold,
                'embedding_model': self.embedding_generator.model_name,
                'embedding_dimension': len(embedding),
                'filename': filename,
                'file_type': file_type,
                'file_size': file_size
            }

            # Add any additional metadata
            if metadata:
                chunk_metadata.update(metadata)

            chunk_obj = Chunk(
                id=str(uuid.uuid4()),
                document_id=document_id,
                text=chunk.text,
                embedding=embedding,
                metadata=chunk_metadata
            )
            chunk_objects.append(chunk_obj)

        # Store in database using pgvector
        print("Inserting chunks into database using pgvector...")
        await self.vector_store.add_chunks(chunk_objects)

        print(
            f"Successfully processed {filename} -> Document ID: {document_id}")
        return document_id

    async def search_documents(self, query: str, limit: int = 5, threshold: float = 0.7,
                               document_ids: Optional[List[str]] = None) -> List[Dict]:
        """
        Search for relevant document chunks using pgvector.
        Args:
            query: Search query
            limit: Maximum number of results
            threshold: Similarity threshold
            document_ids: Optional list of document IDs to filter by

        Returns:
            List of relevant chunks
        """
        query_embedding = self.embedding_generator.embed_text(query)
        return await self.vector_store.search_similar_chunks(
            query_embedding, limit, threshold, document_ids
        )

    async def delete_document(self, document_id: str) -> int:
        """Delete all chunks for a document."""
        return await self.vector_store.delete_document_chunks(document_id)

    async def get_stats(self) -> Dict[str, Any]:
        """Get vector store statistics."""
        return await self.vector_store.get_collection_stats()


embedding_model = 'all-MiniLM-L6-v2'
embedding_generator = EmbeddingGenerator(embedding_model)

In [ ]:
import logfire
import google.generativeai as genai
from pydantic_ai import Agent
from pydantic_ai.models.google import GoogleModel
from pydantic_ai.providers.google import GoogleProvider
from file_validator import FileValidator, FileValidationConfig
from models.models import QueryRequest, UploadResponse, RAGResponse, SimpleRAGResponse, RAGSource, RAGResponseMetadata

# Constants
DEFAULT_TABLE_NAME = "document_chunks"
DEFAULT_EMBEDDING_MODEL = "all-MiniLM-L6-v2"
DEFAULT_CHUNKING_SIMILARITY = 0.5
ALLOWED_CONTENT_TYPES = [
    'application/pdf', 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'text/plain']

# Global configuration

class AppConfig:
    def __init__(self):
        # Initialize logfire with token from environment
        logfire_token = os.getenv('LOGFIRE_WRITE_TOKEN')
        if logfire_token:
            logfire.configure(token=logfire_token)
            print("✓ Logfire configured successfully")
        else:
            print("⚠️ LOGFIRE_WRITE_TOKEN not found, using default configuration")
            logfire.configure()

        self.db_params = {
            'host': os.getenv('DB_HOST', 'localhost'),
            'port': os.getenv('DB_PORT', '5432'),
            'dbname': os.getenv('POSTGRES_DB', 'rag_db'),
            'user': os.getenv('POSTGRES_USER', 'admin'),
            'password': os.getenv('POSTGRES_PASSWORD', 'admin')
        }
        self.file_validator = FileValidator(FileValidationConfig())
        self.agent = self._configure_pydantic_ai()
        self.pipeline = None
        self.reranker = None  # Lazy initialization to avoid startup overhead

    def _configure_pydantic_ai(self):
        gemini_key = os.getenv('GOOGLE_API_KEY')
        if gemini_key:
            try:
                # Configure Pydantic AI Agent with GoogleProvider
                provider = GoogleProvider(api_key=gemini_key)
                model = GoogleModel('gemini-2.5-flash', provider=provider)

                # Create agent with system prompt and output type
                agent = Agent(
                    model,
                    output_type=SimpleRAGResponse,
                    system_prompt="""You are a helpful RAG (Retrieval-Augmented Generation) assistant.
                    Based on the provided context from document search, provide comprehensive answers to user questions.

                    Instructions:
                    - Answer directly and accurately based on the provided context
                    - If the context doesn't fully answer the question, clearly state what information is available
                    - Cite specific sources when making claims, including page numbers when available (e.g., "according to Source 1, Page 5")
                    - Be concise but thorough
                    - Provide a confidence score (0-1) based on how well the context answers the question

                    Respond with:
                    - answer: Your comprehensive response with page references
                    - confidence: Float between 0-1 indicating confidence in the answer
                    - word_count: Number of words in your answer
                    - sources_used: Number of sources used (will be provided)
                    - metadata: Any additional relevant information"""
                )

                # Test the agent with a simple query
                print("✓ Pydantic AI Agent configured successfully")
                return agent
            except Exception as e:
                print(f"❌ Pydantic AI configuration failed: {e}")
                # Fallback to direct genai for backward compatibility
                try:
                    genai.configure(api_key=gemini_key)
                    print("✓ Fallback to direct Gemini API")
                except Exception as fallback_error:
                    print(f"❌ Gemini fallback also failed: {fallback_error}")
        return None

config = AppConfig()

✓ Logfire configured successfully
✓ Pydantic AI Agent configured successfully


Logfire project URL: ]8;id=776459;https://logfire-us.pydantic.dev/adrien-2025/rag-demo\https://logfire-us.pydantic.dev/adrien-2025/rag-demo]8;;\


In [ ]:
# Function test
async def get_pipeline(table_name: str = DEFAULT_TABLE_NAME):
    if config.pipeline is None or config.pipeline.vector_store.table_name != table_name:
        config.pipeline = ChunkEmbeddingPipeline(
            db_params=config.db_params,
            embedding_model=DEFAULT_EMBEDDING_MODEL,
            table_name=table_name
        )
        # Initialize the database for the new pipeline
        await config.pipeline.vector_store._initialize_database()
    return config.pipeline

pipeline = await get_pipeline(table_name)

table_name = 'test1'
initial_limit = 20
threshold = 0.5 
document_ids = None
query = 'What is NLTK?'

results = await pipeline.search_documents(
    query=query,
    limit=initial_limit,
    threshold=threshold,
    document_ids=document_ids
)

In [16]:
len(results)

19

In [25]:
results[0]

{'chunk_id': '6250bf1e-e76e-4bb9-b33b-d05eb07aa77c',
 'text': 'You should now be equipped to work with large datasets, to create robust models of\nlinguistic phenomena, and to extend them into components for practical language\ntechnologies. We hope that the Natural Language Toolkit (NLTK) has served to open\nup the exciting endeavor of practical natural language processing to a broader audience\nthan before.\nIn spite of all that has come before, language presents us with far more than a temporary\n',
 'metadata': {'filename': 'NLTK.pdf',
  'end_index': 1003848,
  'file_size': 3580937,
  'file_type': 'pdf',
  'chunk_size': 768,
  'chunk_index': 5268,
  'page_number': 463,
  'start_index': 1003412,
  'token_count': 83,
  'content_type': 'application/pdf',
  'embedding_model': 'all-MiniLM-L6-v2',
  'upload_timestamp': '139785087795402850',
  'validation_passed': True,
  'embedding_dimension': 384,
  'similarity_threshold': 0.5},
 'document_id': 'b715603a-d540-4474-87dc-62f26a56436d',
 '

In [22]:
from rank_bm25 import BM25Okapi
import numpy as np

# Tokenize the text (simple word splitting)
tokenized_corpus = [doc['text'].lower().split() for doc in results]

# Step 2: Create BM25 index
bm25 = BM25Okapi(tokenized_corpus)

# Step 3: Search with BM25
query = "natural language processing toolkit"
tokenized_query = query.lower().split()

# Get BM25 scores for all documents
bm25_scores = bm25.get_scores(tokenized_query)

# Get top k results
top_k = 5
top_indices = np.argsort(bm25_scores)[::-1][:top_k]

# Retrieve top documents
bm25_results = []
for idx in top_indices:
    result = results[idx].copy()
    result['bm25_score'] = float(bm25_scores[idx])
    bm25_results.append(result)

print("BM25 Results:")
for result in bm25_results:
    print(f"Score: {result['bm25_score']:.4f}")
    print(f"Text: {result['text'][:100]}...")
    print()

BM25 Results:
Score: 8.8590
Text: You should now be equipped to work with large datasets, to create robust models of
linguistic phenom...

Score: 1.7753
Text: 80 | Chapter 3:  Processing Raw Text
Notice that NLTK was needed for tokenization, but not for any o...

Score: 1.4878
Text: opening a URL and reading it into a string. If we now take the further step of creating
an NLTK 
tex...

Score: 0.0000
Text: To perform the first three tasks, we can define a function that simply connects together
NLTK’s defa...

Score: 0.0000
Text: saw in Chapter 1, along with the regular list operations, such as slicing:
>>> text = nltk.Text(toke...



In [26]:
bm25_results

[{'chunk_id': '6250bf1e-e76e-4bb9-b33b-d05eb07aa77c',
  'text': 'You should now be equipped to work with large datasets, to create robust models of\nlinguistic phenomena, and to extend them into components for practical language\ntechnologies. We hope that the Natural Language Toolkit (NLTK) has served to open\nup the exciting endeavor of practical natural language processing to a broader audience\nthan before.\nIn spite of all that has come before, language presents us with far more than a temporary\n',
  'metadata': {'filename': 'NLTK.pdf',
   'end_index': 1003848,
   'file_size': 3580937,
   'file_type': 'pdf',
   'chunk_size': 768,
   'chunk_index': 5268,
   'page_number': 463,
   'start_index': 1003412,
   'token_count': 83,
   'content_type': 'application/pdf',
   'embedding_model': 'all-MiniLM-L6-v2',
   'upload_timestamp': '139785087795402850',
   'validation_passed': True,
   'embedding_dimension': 384,
   'similarity_threshold': 0.5},
  'document_id': 'b715603a-d540-4474-87dc

In [34]:
def rerank_bm25(query:str,
                sources: List[Dict], 
                top_k:int):
    """
    Using bm25 to return top k sources from the inputs
    Args:
        sources: 
            - List of sources from postgresql
            - Format: {'chunk_id': '6250bf1e-e76e-4bb9-b33b-d05eb07aa77c',
                        'text': 'abc'
                        'metadata': {},
                        ...}
        top_k: top result
    """
    # Tokenize the text (simple word splitting)
    tokenized_corpus = [doc['text'].lower().split() for doc in sources]

    # Step 2: Create BM25 index
    bm25 = BM25Okapi(tokenized_corpus)

    # Step 3: Search with BM25
    tokenized_query = query.lower().split()

    # Get BM25 scores for all documents
    bm25_scores = bm25.get_scores(tokenized_query)

    # Get top k results
    top_indices = np.argsort(bm25_scores)[::-1][:top_k]

    # Retrieve top documents
    bm25_results = []
    for idx in top_indices:
        result = sources[idx].copy()
        result['bm25_score'] = float(bm25_scores[idx])
        bm25_results.append(result)

    return bm25_results

query = 'What is NLTK?'

test = rerank_bm25(query, results, top_k=2)
len(test)

2

In [35]:
test[0]

{'chunk_id': 'a82fbcec-4ccc-4765-bdaf-bca4fa128d46',
 'text': '... "it means just what I choose it to mean - neither more nor less."\'\'\'\n>>> [w.lower() for w in nltk.word_tokenize(text)]\n[\'"\', \'when\', \'i\', \'use\', \'a\', \'word\', \',\', \'"\', \'humpty\', \'dumpty\', \'said\', ...]\n',
 'metadata': {'filename': 'NLTK.pdf',
  'end_index': 339553,
  'file_size': 3580937,
  'file_type': 'pdf',
  'chunk_size': 768,
  'chunk_index': 1820,
  'page_number': 159,
  'start_index': 339348,
  'token_count': 99,
  'content_type': 'application/pdf',
  'embedding_model': 'all-MiniLM-L6-v2',
  'upload_timestamp': '139785087795402850',
  'validation_passed': True,
  'embedding_dimension': 384,
  'similarity_threshold': 0.5},
 'document_id': 'b715603a-d540-4474-87dc-62f26a56436d',
 'similarity': 0.5444937997452974,
 'bm25_score': 1.9189252625506943}

In [ ]:
# Convert reranked results back to the expected format
results = [
    {
        'chunk_id': r.get('chunk_id'),
        'text': r.get('text'),
        'document_id': r.get('document_id'),
        'metadata': r.get('metadata'),
        'similarity': r.get('similarity'),
        'rerank_score': r.get('bm25_score')
    }
    for r in test
]

avg_rerank_score = sum(r['rerank_score'] for r in results) / len(results) if results else None

AttributeError: 'dict' object has no attribute 'text'

In [1]:
28500000*0.85

24225000.0

# Processs PDF

In [ ]:
import pymupdf
import fitz  # PyMuPDF
import pandas as pd
import os

def parse_and_format_pdf(file_path):
    # Open the document
    doc = fitz.open(file_path)
    print(f"Processing: {os.path.basename(file_path)} | Total Pages: {len(doc)}\n")
    for page_num in range(len(doc)):
        page = doc[page_num]
        print(f"--- [ PAGE {page_num + 1} ] ---")
        # --- 1. HEADERS & SECTIONS (based on Font Size) ---
        # We use .get_text("dict") to get metadata like font size and style
        blocks = page.get_text("dict")["blocks"]
        for b in blocks:
            if "lines" in b:
                for l in b["lines"]:
                    for s in l["spans"]:
                        text = s["text"].strip()
                        size = s["size"]
                        # Logic: Larger fonts are typically Headers/Sections
                        if text:
                            if size >= 18:
                                print(f"[Header H1] {text}")
                            elif size >= 14:
                                print(f"[Section H2] {text}")
                            elif size >= 12 and "Bold" in s["font"]:
                                print(f"[Sub-Section] {text}")
        # --- 2. TABLES (PyMuPDF Table Finder) ---
        # Requires PyMuPDF v1.21.0+
        tabs = page.find_tables()
        if tabs.tables:
            print(f"\n[Tables Found: {len(tabs.tables)}]")
            for i, tab in enumerate(tabs.tables):
                # Convert to DataFrame for pretty formatting
                df = tab.to_pandas()
                print(f"Table {i+1} Data:\n{df.to_string(index=False)}\n")
        # --- 3. IMAGES ---
        images = page.get_images(full=True)
        if images:
            print(f"\n[Images Found: {len(images)}]")
            for img_index, img in enumerate(images):
                xref = img[0]
                base_image = doc.extract_image(xref)
                image_ext = base_image["ext"]
                # You can save the image or just report its presence
                # filename = f"page{page_num+1}_img{img_index}.{image_ext}"
                # with open(filename, "wb") as f: f.write(base_image["image"])
                print(f"Image {img_index+1}: Detected (Type: {image_ext.upper()}, Xref: {xref})")
        
        print("-" * 30 + "\n")

In [33]:
import fitz  # PyMuPDF
import pandas as pd

def process_page_to_markdown(doc: fitz.Document, page_num: int) -> str:
    page = doc[page_num]
    
    # 1. IDENTIFY TABLES
    tables = page.find_tables()
    table_bboxes = [tab.bbox for tab in tables]
    
    # 2. EXTRACT BLOCKS
    blocks = page.get_text("dict", flags=fitz.TEXT_PRESERVE_IMAGES)["blocks"]
    
    items = []
    
    for block in blocks:
        block_bbox = block["bbox"]
        block_rect = fitz.Rect(block_bbox)
        
        # Check table intersection
        is_in_table = False
        for tab_bbox in table_bboxes:
            tab_rect = fitz.Rect(tab_bbox)
            
            # Use & operator for intersection
            intersection = tab_rect & block_rect
            
            # Calculate areas
            block_area = abs(block_rect.x1 - block_rect.x0) * abs(block_rect.y1 - block_rect.y0)
            inter_area = abs(intersection.x1 - intersection.x0) * abs(intersection.y1 - intersection.y0)
            
            if block_area > 0 and (inter_area / block_area) > 0.5:
                is_in_table = True
                break
        
        if is_in_table:
            continue
            
        # [Rest of the logic remains the same...]
        if block["type"] == 0:  # Text
            block_text = ""
            max_size = 0
            for line in block["lines"]:
                for span in line["spans"]:
                    block_text += span["text"] + " "
                    if span["size"] > max_size:
                        max_size = span["size"]
            
            block_text = block_text.strip()
            if not block_text: continue

            if max_size > 18: content = f"# {block_text}"
            elif max_size > 14: content = f"## {block_text}"
            else: content = block_text
            
            items.append({"y0": block_rect.y0, "x0": block_rect.x0, "type": "text", "content": content})

        elif block["type"] == 1: # Image
            items.append({"y0": block_rect.y0, "x0": block_rect.x0, "type": "image", "content": "<image> image <image>"})

    # Add Tables
    for table in tables:
        df = table.to_pandas()
        if not df.empty:
            markdown = df.to_markdown(index=False)
            items.append({"y0": table.bbox[1], "x0": table.bbox[0], "type": "table", "content": f"<table>\n{markdown}\n<table>"})

    items.sort(key=lambda x: (x["y0"], x["x0"]))
    return items

In [ ]:
file_path = r"D:\Books\2-Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O’Reilly-Media-2019.pdf"
doc = fitz.open(file_path)

In [51]:
doc.page_count

510

In [59]:
page_num = 500
items = process_page_to_markdown(doc, page_num)
# Build final string
markdown_output = f"[Page {page_num + 1}]\n\n"
for item in items:
    markdown_output += item["content"] + "\n\n"

output_file = "output_markdown.md"
print(f"Writing output to {output_file}...")
with open(output_file, "w", encoding="utf-8") as f:
    f.write(markdown_output)

Writing output to output_markdown.md...


In [49]:
items

[{'y0': 59.25000762939453,
  'x0': 79.44999694824219,
  'type': 'image',
  'content': '<image> image <image>'},
 {'y0': 163.25164794921875,
  'x0': 72.0,
  'type': 'text',
  'content': 'Figure 3-3. Decision threshold and precision/recall tradeoff'},
 {'y0': 187.98814392089844,
  'x0': 71.99626159667969,
  'type': 'text',
  'content': 'Scikit-Learn does not let you set the threshold directly, but it does give you access to the decision scores that it uses to make predictions. Instead of calling the classifier’s predict()  method, you can call its  decision_function()  method, which returns a score for each instance, and then make predictions based on those scores using any threshold you want:'},
 {'y0': 260.1324462890625,
  'x0': 88.9982681274414,
  'type': 'text',
  'content': '>>>  y_scores   =   sgd_clf . decision_function ([ some_digit ]) >>>  y_scores array([2412.53175101]) >>>  threshold   =   0 >>>  y_some_digit_pred   =  ( y_scores   >   threshold ) array([ True])'},
 {'y0': 325

In [38]:
markdown_output

'[Page 121]\n\ntreats all values equally, the harmonic mean gives much more weight to low values. As a result, the classifier will only get a high F 1  score if both recall and precision are high.\n\nEquation 3-3. F 1\n\nF 1  = 2 1 precision   + 1 recall = 2 ×   precision × recall precision + recall   = TP\n\nTP  +   FN  +  FP 2\n\nTo compute the F 1  score, simply call the  f1_score()  function:\n\n>>>  from   sklearn.metrics   import   f1_score >>>  f1_score ( y_train_5 ,  y_train_pred ) 0.7420962043663375\n\nThe F 1  score favors classifiers that have similar precision and recall. This is not always what you want: in some contexts you mostly care about precision, and in other con‐ texts you really care about recall. For example, if you trained a classifier to detect vid‐ eos that are safe for kids, you would probably prefer a classifier that rejects many good videos (low recall) but keeps only safe ones (high precision), rather than a clas‐ sifier that has a much higher recall but l

In [39]:
# for page_num in range(len(doc)):
#     page = doc[page_num]
#     print(f"--- [ PAGE {page_num + 1} ] ---")
#     # --- 1. HEADERS & SECTIONS (based on Font Size) ---
#     # We use .get_text("dict") to get metadata like font size and style
#     blocks = page.get_text("dict")["blocks"]
#     for b in blocks:
#         if "lines" in b:
#             for l in b["lines"]:
#                 for s in l["spans"]:
#                     text = s["text"].strip()
#                     size = s["size"]
#                     # Logic: Larger fonts are typically Headers/Sections
#                     if text:
#                         if size >= 18:
#                             print(f"[Header H1] {text}")
#                         elif size >= 14:
#                             print(f"[Section H2] {text}")
#                         elif size >= 12 and "Bold" in s["font"]:
#                             print(f"[Sub-Section] {text}")

In [ ]:
# for page_num in range(len(doc)):
#     page = doc[page_num]
#     tabs = page.find_tables()
#     if tabs.tables:
#         print(f"\n[Tables Found: {len(tabs.tables)}] in {page_num}")
#         for i, tab in enumerate(tabs.tables):
#             # Convert to DataFrame for pretty formatting
#             df = tab.to_pandas()
#             print(f"Table {i+1} Data:\n{df.to_string(index=False)}\n")

page = doc[page_num]

# 1. IDENTIFY TABLES
# ------------------
# We find tables first to get their bounding boxes.
# This helps us avoid duplicating text that is already inside a table.
tables = page.find_tables()
table_bboxes = [tab.bbox for tab in tables]

# 2. EXTRACT ALL BLOCKS (Text & Images)
# -------------------------------------
# get_text("dict") provides detailed layout info.
# flags=fitz.TEXT_PRESERVE_IMAGES ensures image blocks are included.
blocks = page.get_text("dict", flags=fitz.TEXT_PRESERVE_IMAGES)["blocks"]
blocks

[{'type': 0,
  'number': 0,
  'flags': 0,
  'bbox': (71.99696350097656,
   52.29900360107422,
   432.0000915527344,
   91.66349792480469),
  'lines': [{'spans': [{'size': 10.5,
      'flags': 4,
      'bidi': 0,
      'char_flags': 16,
      'font': 'MinionPro-Regular',
      'color': 0,
      'alpha': 255,
      'ascender': 0.9890000224113464,
      'descender': -0.36000001430511475,
      'text': 'treats all values equally, the harmonic mean gives much more weight to low values.',
      'origin': (72.0, 62.683502197265625),
      'bbox': (72.0, 52.29900360107422, 432.0000915527344, 66.4635009765625)}],
    'wmode': 0,
    'dir': (1.0, 0.0),
    'bbox': (72.0, 52.29900360107422, 432.0000915527344, 66.4635009765625)},
   {'spans': [{'size': 10.5,
      'flags': 4,
      'bidi': 0,
      'char_flags': 16,
      'font': 'MinionPro-Regular',
      'color': 0,
      'alpha': 255,
      'ascender': 0.9890000224113464,
      'descender': -0.36000001430511475,
      'text': 'As a result, the 

In [47]:
blocks[0]['lines'][0]['spans']

[{'size': 10.5,
  'flags': 4,
  'bidi': 0,
  'char_flags': 16,
  'font': 'MinionPro-Regular',
  'color': 0,
  'alpha': 255,
  'ascender': 0.9890000224113464,
  'descender': -0.36000001430511475,
  'text': 'treats all values equally, the harmonic mean gives much more weight to low values.',
  'origin': (72.0, 62.683502197265625),
  'bbox': (72.0, 52.29900360107422, 432.0000915527344, 66.4635009765625)}]